# M1 — Data understanding (Zidio FORESIGHT)

NorthBay Living provides **four extracts**:

1. `sales_daily`
2. `sku_master`
3. `calendar`
4. `inventory_snapshots`

This notebook inspects schema, missing values, duplicates, and SKU coverage before cleaning.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline import load_raw, clean_datasets, build_quality_report

raw = load_raw()
clean = clean_datasets(raw)
report = build_quality_report(raw, clean)

for name, df in raw.items():
    print(f"\n=== {name} (raw) ===")
    print("Shape:", df.shape)
    print(df.dtypes)
    print("Missing per column:\n", df.isna().sum())
    print(df.head(3))


=== sales_daily (raw) ===
Shape: (36550, 6)
Date              str
SKU               str
Units_Sold      int64
Revenue       float64
Price         float64
Promotion       int64
dtype: object
Missing per column:
 Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64
         Date     SKU  Units_Sold   Revenue    Price  Promotion
0  2024-01-01  SKU001           5  18320.25  3664.05          0
1  2024-01-01  SKU002          15  57085.35  3805.69          0
2  2024-01-01  SKU003           5  40391.15  8078.23          0

=== sku_master (raw) ===
Shape: (50, 8)
SKU                          str
Product_Name                 str
Category                     str
Subcategory                  str
Launch_Date                  str
Cost_Price               float64
Selling_Price            float64
Gross_Margin_Per_Unit    float64
dtype: object
Missing per column:
 SKU                      0
Product_Name             0
Category                 0
Su

## Duplicate keys (assignment data-quality checks)

In [2]:
sales_dup = raw["sales_daily"].duplicated(subset=["Date", "SKU"]).sum()
inv_dup = raw["inventory_snapshots"].duplicated(subset=["Snapshot_Date", "SKU"]).sum()
print("Duplicate (Date, SKU) in sales:", sales_dup)
print("Duplicate (Snapshot_Date, SKU) in inventory:", inv_dup)

Duplicate (Date, SKU) in sales: 0
Duplicate (Snapshot_Date, SKU) in inventory: 0


## SKU alignment across tables

In [3]:
sales_skus = set(clean["sales_daily"]["SKU"].unique())
inv_skus = set(clean["inventory_snapshots"]["SKU"].unique())
master_skus = set(clean["sku_master"]["SKU"].unique())

print("SKUs in sales:", len(sales_skus))
print("SKUs in inventory snapshots:", len(inv_skus))
print("SKUs in master:", len(master_skus))
print("Sales ∩ inventory:", len(sales_skus & inv_skus))
print("Analysis scope: 50 SKUs with sales history (per brief)")

SKUs in sales: 50
SKUs in inventory snapshots: 200
SKUs in master: 50
Sales ∩ inventory: 50
Analysis scope: 50 SKUs with sales history (per brief)


## Persist M1 outputs (processed CSVs + quality report)

In [4]:
from src.pipeline import run_data_foundation

clean, report = run_data_foundation(persist=True)
print("Data quality summary:")
for k, v in report.items():
    print(f"  {k}: {v}")

Data quality summary:
  sales_daily_rows: 36550
  inventory_rows: 4800
  sku_master_rows: 50
  calendar_rows: 731
  sales_duplicate_date_sku_removed: 0
  inventory_duplicate_snapshot_sku_removed: 0
  sales_skus: 50
  inventory_skus: 200
  skus_in_sales_and_inventory: 50
  inventory_skus_not_in_sales: 150
  calendar_missing_holiday: 723
  calendar_missing_promotion_event: 656
